In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
from utils import describe_data, get_frequency_table, plot_df_chart, load_report_data

In [3]:
DICT = {
    "FEDFUNDS": {
        "title": "Federal Funds Effective Rate",
        "unit": "Percent"
    },
    "M2SL": {
        "title": "M2 ",
        "unit": "Billions of Dollars"
    },
    "WALCL": {
        "title": "Assets: Total Assets: Total Assets (Less Eliminations from Consolidation): Wednesday Level",
        "unit": "Millions of U.S. Dollars"
    },
    "GFDEBTN": {
        "title": "Federal Debt: Total Public Debt",
        "unit": "Millions of U.S. Dollars"
    },
    "WGS10YR": {
        "title": "Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity, Quoted on an Investment Basis",
        "unit": "Percent"
    },
    "T10Y2Y": {
        "title": "10-Year Treasury Constant Maturity Minus 2-Year Treasury Constant Maturity",
        "unit": "Percent"
    },
    "FYFR": {
        "title": "Federal Receipts",
        "unit": "Millions of Dollars"
    },
    "FYONET": {
        "title": "Federal Net Outlays",
        "unit": "Millions of Dollars"
    },
}

In [4]:
INDICATOR = 'FYONET'

MoM_col = 'MoM ^ %'
YoY_col = 'YoY ^ %'

COL = YoY_col

In [5]:
df = load_report_data(INDICATOR)

# df[COL] = df[INDICATOR].pct_change(fill_method=None) * 100
df[COL] = (df[INDICATOR].ffill().pct_change() * 100)

df.dropna(inplace=True)

df.tail()

# df[COL] = df[INDICATOR].pct_change(fill_method=None, periods=12 if COL == YoY_col else 1) * 100
df[COL] = df[INDICATOR].ffill().pct_change(periods=12 if COL == YoY_col else 1) * 100

df.tail()

,DATE,FYONET,YoY ^ %
120,2021-09-30,6822461,93.947909
121,2022-09-30,6273259,81.461257
122,2023-09-30,6134526,70.258544
123,2024-09-30,6734896,90.976200
124,2025-09-30,7009974,102.900592


- **Plot data**

In [6]:
plot_df_chart(
    df[[INDICATOR]],
    chart_title = DICT[INDICATOR]["title"],
    yaxis_title = DICT[INDICATOR]["unit"],
    use_markers=False,
    fill=True
)

- **% change**

In [ ]:
import plotly.graph_objects as go

fig = plot_df_chart(
    df[[COL]],
    chart_title = DICT[INDICATOR]["title"] + f' ({COL})',
    chart_type = "bar",
    yaxis_title = "%",
    draw=False,
)

AVG_PERIOD = 12 * 5

fig.add_trace(go.Scatter(
    x=df.index,
    y=df[COL].rolling(window=AVG_PERIOD).mean(),
    name='Average ' + COL + f' ({AVG_PERIOD})',
    line=dict(
        width=1,
        color="#F2C14E",
        # dash="dash"
    ),
    # hoverinfo="skip",
    # showlegend=False,
))

fig.show()

- **Stats**

In [8]:
df_stats, stats = describe_data(df[COL].dropna())

df_stats

,Value
Metric,
nobs,112
Min %,-80.657546
Max %,3093.955095
Mean %,271.165871
Median %,138.615739
Mode %,-80.657546
Variance,216880.106175
Skewness,3.905147
Kurtosis,16.447738


- **Freq**

In [9]:
freq_table = get_frequency_table(df.dropna()[COL])

freq_table

,Frequency,Probability %,Cumulative Probability %
Less than 448.44%,100,89.285714,89.285714
448.44% to 977.55%,7,6.250000,95.535714
977.55% to 1506.65%,0,0.000000,95.535714
1506.65% to 2035.75%,2,1.785714,97.321429
2035.75% to 2564.85%,2,1.785714,99.107143
Greater than 2564.85%,1,0.892857,100.000000


In [10]:
fig = plot_df_chart(
    freq_table[['Probability %']],
    chart_title = f'{DICT[INDICATOR]['title']} (% change frequency)',
    chart_type = "bar",
    yaxis_title = "Probability %",
    width=1000,
    height=700,
    show_rangeslider=False
)